# Malware Classification Using Static Analysis

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# --- Synthetic dataset creation ---

np.random.seed(42)

# Number of samples and classes
n_samples = 1000
classes = ['Trojan', 'Worm', 'Adware', 'Ransomware', 'Spyware']

# Numerical features (e.g., file size, entropy, opcode frequency)
file_size = np.random.randint(10000, 500000, n_samples)
entropy = np.random.uniform(1.0, 8.0, n_samples)
opcode_freq = np.random.rand(n_samples)

# Categorical feature (e.g., section names)
sections = np.random.choice(['.text', '.data', '.rdata', '.edata', '.rsrc'], n_samples)

# Labels (malware family)
labels = np.random.choice(classes, n_samples)

# Build DataFrame
df = pd.DataFrame({
    'file_size': file_size,
    'entropy': entropy,
    'opcode_freq': opcode_freq,
    'section': sections,
    'label': labels
})

# --- Preprocessing ---

# Encode categorical features (one-hot encoding for 'section')
df_encoded = pd.get_dummies(df, columns=['section'])

# Separate features and labels
X = df_encoded.drop('label', axis=1)
y = df_encoded['label']

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

# --- Modeling ---

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

# --- Evaluation ---

print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()
